# 01 Spike: can Lonboard carry kharita's legend?

Throwaway notebook. Nothing here is kept except the answers in the last cell.

**Questions to answer:**

1. Do 1 px outlines show with `line_width_units="pixels"`?
2. Does the hover highlight colour work?
3. Can a legend sit on top of the map in the notebook?
4. Does the legend survive `to_html()` and show in a browser? **This is the gate.**
5. Does a keyless basemap (OpenFreeMap) load?
6. Do tooltips work with only chosen columns?
7. Does extrusion work with a tilted camera?
8. How long does a 1 million point layer take?

In [25]:
import time
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap

from kharita.palette import load_colors, load_ramps

# Lonboard's pandas deprecation warnings are not ours
warnings.filterwarnings("ignore", module="lonboard")

OUT = Path("../spike_out")
OUT.mkdir(exist_ok=True)
ATTRIBUTION = "Boundaries and population: U.S. Census Bureau, via HDX, CC BY 4.0. Simplified and joined for this demo."

states = gpd.read_file("../data/processed/sudan_states.gpkg")
print(len(states), states.crs)
states.head(3)

18 EPSG:4326


,key,pop,area_km2,density,geometry
0,SDN_01,832112.0,37887.377919,21.962776,"MULTIPOLYGON (((34.45048 12.56224, 34.56863 12..."
1,SDN_02,553515.0,31116.294817,17.788590,"MULTIPOLYGON (((23.49428 13.55935, 23.62734 13..."
2,SDN_03,1216719.0,52982.531290,22.964531,"MULTIPOLYGON (((25.76281 12.77404, 25.82233 12..."


In [26]:
def hex_to_rgba(hex_color, alpha=255):
    # '#abc' or '#aabbcc' to [r, g, b, a]
    h = hex_color.lstrip("#")
    if len(h) == 3:
        h = "".join(c * 2 for c in h)
    return [int(h[i : i + 2], 16) for i in (0, 2, 4)] + [alpha]


def quantile_colors(values, ramp_name="ylorrd", k=5, null_hex="#d1d5db"):
    # Quantile classes, one colour per class, grey for missing
    values = np.asarray(values, dtype=float)
    ok = np.isfinite(values)
    edges = np.quantile(values[ok], np.linspace(0, 1, k + 1))
    classes = np.searchsorted(edges[1:-1], values, side="right")
    ramp = load_ramps()[ramp_name]["colors"]
    picks = [ramp[i] for i in np.linspace(0, len(ramp) - 1, k).round().astype(int)]
    lut = np.array([hex_to_rgba(c) for c in picks], dtype=np.uint8)
    out = lut[np.clip(classes, 0, k - 1)]
    out[~ok] = hex_to_rgba(null_hex)
    counts = [int(((classes == i) & ok).sum()) for i in range(k)]
    return out, edges, picks, counts, int((~ok).sum())


fill, edges, picks, counts, n_null = quantile_colors(states["density"])
print("breaks:", np.round(edges, 1))
print("counts:", counts, "missing:", n_null)

breaks: [  1.9   9.5  16.7  24.9  38.3 248.9]
counts: [4, 3, 4, 3, 4] missing: 0


In [27]:
# Q1 and Q2: pixel outlines and hover highlight on the default basemap
layer = PolygonLayer.from_geopandas(
    states[["geometry"]],
    get_fill_color=fill,
    get_line_color=hex_to_rgba(load_colors()["white"]),
    get_line_width=1,
    line_width_units="pixels",
    pickable=True,
    auto_highlight=True,
    highlight_color=hex_to_rgba(load_colors()["amber"]["400"], 200),
)
m = Map(layer, basemap=MaplibreBasemap(style=CartoStyle.PositronNoLabels))
m

In [28]:
def legend_html(title, edges, colors, counts, n_null, null_hex="#d1d5db", fmt="{:,.1f}", note=""):
    # Self-contained legend: inline styles only so it works in any page
    rows = []
    for i, c in enumerate(colors):
        label = f"{fmt.format(edges[i])} to {fmt.format(edges[i + 1])}"
        rows.append(
            f'<div style="display:flex;align-items:center;gap:8px;margin:2px 0">'
            f'<span style="width:14px;height:14px;background:{c};border:1px solid #0002;display:inline-block"></span>'
            f'<span>{label}</span><span style="color:#6b7280;margin-left:auto">{counts[i]}</span></div>'
        )
    if n_null:
        rows.append(
            f'<div style="display:flex;align-items:center;gap:8px;margin:2px 0">'
            f'<span style="width:14px;height:14px;background:{null_hex};border:1px solid #0002;display:inline-block"></span>'
            f'<span>No data</span><span style="color:#6b7280;margin-left:auto">{n_null}</span></div>'
        )
    foot = f'<div style="margin-top:6px;font-size:10px;color:#6b7280;max-width:220px">{note}</div>' if note else ""
    return (
        '<div class="kharita-legend" style="font:12px/1.3 system-ui,sans-serif;color:#111827;'
        'background:#ffffffee;padding:10px 12px;border-radius:6px;box-shadow:0 1px 4px #0003;min-width:170px">'
        f'<div style="font-weight:600;margin-bottom:6px">{title}</div>{"".join(rows)}{foot}</div>'
    )


LEGEND = legend_html("People per km2 (quantiles)", edges, picks, counts, n_null, note=ATTRIBUTION)
widgets.HTML(LEGEND)

HTML(value='<div class="kharita-legend" style="font:12px/1.3 system-ui,sans-serif;color:#111827;background:#ff…

In [29]:
# Q3 attempt A: legend overlaid on the map with a CSS grid
css = widgets.HTML(
    "<style>.kh-overlay{z-index:10;justify-self:end;align-self:end;margin:0 12px 40px 0}</style>"
)
m_a = Map(layer, basemap=MaplibreBasemap(style=CartoStyle.PositronNoLabels))
m_a.layout = widgets.Layout(grid_area="1 / 1 / 2 / 2")
leg_a = widgets.HTML(LEGEND, layout=widgets.Layout(grid_area="1 / 1 / 2 / 2", width="auto"))
leg_a.add_class("kh-overlay")
overlay = widgets.GridBox([m_a, leg_a], layout=widgets.Layout(grid_template_columns="1fr", grid_template_rows="500px"))
display(css, overlay)

HTML(value='<style>.kh-overlay{z-index:10;justify-self:end;align-self:end;margin:0 12px 40px 0}</style>')

GridBox(children=(<lonboard._map.Map object at 0x00000282C76A6EA0>, HTML(value='<div class="kharita-legend" st…

In [30]:
# Q3 attempt B, the fallback: legend beside the map
m_b = Map(layer, basemap=MaplibreBasemap(style=CartoStyle.PositronNoLabels))
m_b.layout = widgets.Layout(flex="1 1 auto")
widgets.HBox([m_b, widgets.HTML(LEGEND)], layout=widgets.Layout(align_items="flex-end"))

In [31]:
# Q4, the gate: export with the legend injected, then open it in your browser
m_export = Map(layer, basemap=MaplibreBasemap(style=CartoStyle.PositronNoLabels), height="100vh")
page = m_export.to_html(title="kharita spike: Sudan density")
overlay_css = "<style>body{margin:0}.kh-fixed{position:fixed;right:12px;bottom:36px;z-index:1000}</style>"
page = page.replace("</body>", f'{overlay_css}<div class="kh-fixed">{LEGEND}</div></body>', 1)
path = OUT / "sudan_density.html"
path.write_text(page, encoding="utf-8")
print("legend in file:", "kharita-legend" in path.read_text(encoding="utf-8"))
print(round(path.stat().st_size / 1e6, 2), "MB")
webbrowser.open(path.resolve().as_uri())

legend in file: True
4.7 MB


True

In [32]:
# Q5: keyless basemap; change STYLE to positron, bright or liberty and rerun
STYLE = "liberty"
m_free = Map(layer, basemap=MaplibreBasemap(style=f"https://tiles.openfreemap.org/styles/{STYLE}"))
m_free

In [33]:
# Q6: tooltip with only the columns we choose, side panel off
tip_layer = PolygonLayer.from_geopandas(
    states[["key", "pop", "density", "geometry"]],
    get_fill_color=fill,
    get_line_color=[255, 255, 255, 255],
    get_line_width=1,
    line_width_units="pixels",
    pickable=True,
)
Map(tip_layer, show_tooltip=True, show_side_panel=False)

In [34]:
# Q7: extrusion, tallest state 300 km high, camera tilted 45 degrees
dens = states["density"].to_numpy(dtype=float)
height_m = dens / np.nanmax(dens) * 300_000
ext_layer = PolygonLayer.from_geopandas(
    states[["geometry"]],
    get_fill_color=fill,
    extruded=True,
    get_elevation=height_m,
    pickable=True,
    auto_highlight=True,
)
Map(ext_layer, view_state={"longitude": 30, "latitude": 13, "zoom": 4.3, "pitch": 45, "bearing": -10})

In [35]:
# Q8: 1 million random points inside Sudan's bounding box
rng = np.random.default_rng(0)
n = 1_000_000
xmin, ymin, xmax, ymax = states.total_bounds
t0 = time.perf_counter()
pts = gpd.GeoDataFrame(
    {"v": rng.lognormal(size=n)},
    geometry=gpd.points_from_xy(rng.uniform(xmin, xmax, n), rng.uniform(ymin, ymax, n)),
    crs="EPSG:4326",
)
t1 = time.perf_counter()
pfill, *_ = quantile_colors(pts["v"], "viridis", 5)
t2 = time.perf_counter()
pt_layer = ScatterplotLayer.from_geopandas(pts[["geometry"]], get_fill_color=pfill, get_radius=1.5, radius_units="pixels")
t3 = time.perf_counter()
print(f"build frame {t1 - t0:.2f}s, classify and colour {t2 - t1:.2f}s, layer {t3 - t2:.2f}s")
big = Map(pt_layer)
big

build frame 0.81s, classify and colour 0.09s, layer 0.89s


## Results

Fill this in, then paste it back to Claude.

| # | Question | Works? | Notes |
| --- | --- | --- | --- |
| 1 | 1 px outlines | yes | it works |
| 2 | Hover highlight | yes | it works |
| 3A | Legend overlaid in notebook | yes | im able to drag with out an issue |
| 3B | Legend beside map | yes | follback works |
| 4 | **Legend in exported HTML (gate)** | yes | a website works similar to 3a featuers |
| 5 | OpenFreeMap basemap (which styles) | yes | each style provide different basemap |
| 6 | Tooltip with chosen columns | yes | hover works |
| 7 | Extrusion with tilt | yes | 3d map works |
| 8 | 1M points: times printed, map smooth? | no | it just created a box of points that are not only limited to sudan border. build frame 0.99s, classify and colour 0.10s, layer 1.09s|

Also note: are you running this in VS Code or JupyterLab? on vscode